<a href="https://colab.research.google.com/github/SithmiDesilva/Hallucination-Aware-AI-Framework-for-Trustworthy-Supply-Chain-Decision-Intelligence.-/blob/DataCo_Supply_Chain_Preprocessing/DataCo_tokenized_access_logs_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SECTION 1: INITIAL SETUP AND DATA LOADING

In [6]:
# ============================================================================
# SECTION 1: SETUP, INSTALLATION & DATA LOADING
# ============================================================================

# Install required packages
!pip install pandas numpy matplotlib seaborn wordcloud -q

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter
import re
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# File path
file_path = '/content/drive/MyDrive/IRP/Data Co/tokenized_access_logs.csv'

# Load data
try:
    df_raw = pd.read_csv(file_path, encoding='utf-8', low_memory=False)
    print(f"\n Data loaded successfully!")
    print(f"  - Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
    print(f"  - Columns: {list(df_raw.columns)}")
except FileNotFoundError:
    print(f"\n File not found at: {file_path}")
    print("Please upload the file:")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df_raw = pd.read_csv(filename, encoding='utf-8', low_memory=False)
    print(f"\n Data loaded from upload!")
    print(f"  - Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
except Exception as e:
    print(f"\n Error: {e}")
    df_raw = None

# Quick preview
if df_raw is not None:
    print("\n--- First 5 Rows ---")
    display(df_raw.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

 Data loaded successfully!
  - Shape: 469,977 rows × 8 columns
  - Columns: ['Product', 'Category', 'Date', 'Month', 'Hour', 'Department', 'ip', 'url']

--- First 5 Rows ---


,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...
3,Under Armour Men's Compression EV SL Slide,electronics,9/1/2017 6:00,Sep,6,footwear,206.56.112.1,/department/footwear/category/electronics/prod...
4,Pelican Sunstream 100 Kayak,water sports,9/1/2017 6:01,Sep,6,fan shop,136.108.56.242,/department/fan%20shop/category/water%20sports...


## SECTION 2: INITIAL DATA INSPECTION (BEFORE PREPROCESSING)

In [7]:
# 2.1 Data Types & Memory
print("\n--- Data Types & Memory Usage ---")
df_raw.info()


--- Data Types & Memory Usage ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 469977 entries, 0 to 469976
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Product     469977 non-null  object
 1   Category    469977 non-null  object
 2   Date        469977 non-null  object
 3   Month       469977 non-null  object
 4   Hour        469977 non-null  int64 
 5   Department  469977 non-null  object
 6   ip          469977 non-null  object
 7   url         469977 non-null  object
dtypes: int64(1), object(7)
memory usage: 28.7+ MB


In [8]:
# 2.2 Numerical Summary
print("\n--- Numerical Columns Summary ---")
numerical_cols = df_raw.select_dtypes(include=['float64', 'int64']).columns
if len(numerical_cols) > 0:
    display(df_raw[numerical_cols].describe())
else:
    print("No numerical columns found")


--- Numerical Columns Summary ---


,Hour
count,469977.000000
mean,14.591827
std,5.574014
min,0.000000
25%,10.000000
50%,15.000000
75%,20.000000
max,23.000000


In [10]:
# 2.3 Missing Values

missing_df = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Pct': (df_raw.isnull().sum() / len(df_raw)) * 100
}).sort_values('Missing_Pct', ascending=False)
print(missing_df[missing_df['Missing_Count'] > 0])
if len(missing_df[missing_df['Missing_Count'] > 0]) == 0:
    print(" No missing values found")

Empty DataFrame
Columns: [Column, Missing_Count, Missing_Pct]
Index: []
 No missing values found
